# Бонус — как пользоваться своим дообученным адаптером (локально, без GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITrubnikov/Train_of_Thought-homework/blob/main/notebooks/module-7-7-finetuning/use-adapter.ipynb)

> Продолжение основного ноутбука модуля 7.7:
> [notebook.ipynb](https://github.com/ITrubnikov/Train_of_Thought-homework/blob/main/notebooks/module-7-7-finetuning/notebook.ipynb). Там мы обучали адаптер — здесь им пользуемся.

В основном ноутбуке мы дообучили `Qwen2.5-0.5B-Instruct` LoRA-адаптером на фактах
про **вымышленное** устройство «Кьюби» и выложили адаптер на HuggingFace Hub:
[`HOhus/qubi-lora-0_5b`](https://huggingface.co/HOhus/qubi-lora-0_5b).

Здесь — как этот адаптер **подключить и вызвать**. Главное: для инференса GPU не
нужен — модель 0.5B с маленькой LoRA-добавкой спокойно крутится прямо на ноутбуке
(CPU или Apple MPS). Запускается целиком (`Run all`).

## Что вообще произошло (на пальцах)

1. Взяли готовую модель `Qwen2.5-0.5B-Instruct` и **заморозили** её веса.
2. Навесили маленькую LoRA-добавку `W → W + B·A` и обучили **только её** на наших
   фактах про Кьюби (это ~1.75% параметров, файл-адаптер ~35 МБ).
3. Выложили адаптер на HuggingFace Hub.

Адаптер — это **не целая модель**, а «насадка» на базу. Поэтому чтобы им
воспользоваться, нужно: загрузить ту же базовую модель и **подключить к ней
адаптер**. Это и делаем ниже.

## Шаг 1. Пакеты и устройство

In [ ]:
# import torch ДО pip install (на Colab/Kaggle установка пакетов в живой сессии
# иногда ломает сам import torch). Локально это просто грузит torch.
import torch
print("torch", torch.__version__)

# Ставим пакеты БЕЗ -q — чтобы был виден прогресс (иначе кажется, что ноутбук «висит»).
# ipywidgets нужен, чтобы прогресс-бары HuggingFace рендерились прямо в ноутбуке.
print("[1/2] Ставлю пакеты (при первом запуске ~1-2 мин, потом мгновенно)...")
!pip install "transformers>=5.0" "peft==0.19.1" "ipywidgets>=8"
!pip uninstall -y -q torchao 2>/dev/null || true   # на Kaggle ломает peft; локально no-op
print("[1/2] пакеты готовы")

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32   # на MPS/CPU держим fp32
print("device:", DEVICE)

## Шаг 2. Грузим базу и навешиваем адаптер с HF Hub

`PeftModel.from_pretrained(base, ADAPTER)` скачивает адаптер (~35 МБ) и подключает
его к базе. Токенайзер мы тоже залили в репо адаптера — берём его оттуда.

Первый запуск качает базу (~1 ГБ) и адаптер с HuggingFace; дальше всё из кэша.

In [ ]:
BASE    = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER = "HOhus/qubi-lora-0_5b"      # поменяйте на свой репо, если выкладывали под своим ником

print("[2/2] гружу базовую модель (~1 ГБ при первом запуске; прогресс-бары ниже)...")
tok   = AutoTokenizer.from_pretrained(ADAPTER)
base  = AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE).to(DEVICE)
print("[2/2] подключаю адаптер с HF Hub (~35 МБ)...")
model = PeftModel.from_pretrained(base, ADAPTER).to(DEVICE)
model.train(False)   # режим инференса, не обучения

# Системный промпт — ТОТ ЖЕ, что при обучении. Иначе Кьюби-знание проявляется слабее.
SYSTEM = ("Ты — полезный ассистент. Помимо обычных вопросов ты хорошо знаешь "
          "карманный ИИ-помощник «Кьюби» и точно отвечаешь на вопросы о нём.")

@torch.no_grad()
def ask(q, max_new_tokens=64):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("готово, можно спрашивать")

## Шаг 3. Спрашиваем — и сравниваем с «голой» базой

`peft` умеет временно **отключить** адаптер (`disable_adapter()`) — это та же база
без нашей добавки. Удобно увидеть разницу на одной и той же модели: без адаптера
модель про Кьюби фантазирует, с адаптером — отвечает по нашим фактам.

In [ ]:
for q in ["Кто создал Кьюби?", "Сколько стоит Кьюби?", "Какой девиз у Кьюби?"]:
    with model.disable_adapter():        # временно выключаем LoRA -> чистая база
        base_ans = ask(q)
    ft_ans = ask(q)                      # адаптер включён обратно
    print("Q      :", q)
    print("БАЗА   :", base_ans)
    print("АДАПТЕР:", ft_ans, "\n")

## Шаг 4 (опционально). Слить адаптер в одну модель

Если хочется отдать модель целиком (без `peft` при загрузке) — сливаем адаптер в
веса базы и сохраняем как обычную модель. После этого её можно грузить просто через
`AutoModelForCausalLM.from_pretrained("qubi-merged")`.

In [ ]:
# merge_and_unload вшивает B·A в веса базы и убирает обёртку peft.
merged = model.merge_and_unload()
merged.save_pretrained("qubi-merged")
tok.save_pretrained("qubi-merged")
print("слитая модель сохранена в ./qubi-merged — грузится как обычная модель, без peft")
# (после merge переменная model уже без адаптера; для новых ask перезапустите Шаг 2)

## Памятка (на чём легко споткнуться)

- **Системный промпт** должен совпадать с обучающим (см. `SYSTEM` выше) — без него
  выученные факты проявляются слабее.
- **Та же база.** Адаптер обучен под `Qwen2.5-0.5B-Instruct`; навесить его на другую
  модель не получится.
- **dtype на Mac.** На MPS/CPU держите `float32` — fp16 на маках иногда мусорит в
  генерации. На CUDA можно `float16`.
- **Это инференс, не дообучение.** GPU и `bitsandbytes` тут не нужны — отсюда и
  возможность гонять локально.

## Что мы прошли в модуле 7.7 целиком

Обучили адаптер (основной [notebook.ipynb](https://github.com/ITrubnikov/Train_of_Thought-homework/blob/main/notebooks/module-7-7-finetuning/notebook.ipynb)) → выложили на HF Hub → подключили
и вызвали здесь (в т.ч. локально). Это полный цикл «свой fine-tune»: от датасета до
переиспользуемого артефакта. Теория — в [лекции модуля](https://itrubnikov.github.io/Train_of_Thought/docs/modules/07-7-finetuning/).